# Rerank

Cross-encoder reranking: given queries and candidate documents, score each
(query, document) pair and return top-K indices and scores per query.

In [ ]:
#|default_exp utils.rerank

In [ ]:
#|export
import asyncio

import numpy as np

from ai_index.const import rerank_models_config_path
from ai_index.utils._model_config import _resolve_model_args, _split_remote_kwargs

def _apply_instruction(query, instruction):
    """Prepend instruction to query for API rerankers that support it (e.g. Voyage)."""
    if instruction:
        return f"{instruction}. {query}"
    return query


def _apply_instruction_zeroentropy(query, instruction):
    """Embed instruction using ZeroEntropy's XML tag format."""
    if instruction:
        return f"<query>{query}</query> <instruction>{instruction}</instruction>"
    return query


def _rerank_api(queries, documents, top_k, model_name, instruction=""):
    """Rerank using a hosted API (e.g. Voyage, Cohere) via litellm."""
    import litellm
    import time

    n_queries = len(queries)
    all_indices = np.zeros((n_queries, top_k), dtype=np.int64)
    all_scores = np.zeros((n_queries, top_k), dtype=np.float32)

    for qi in range(n_queries):
        q = _apply_instruction(queries[qi], instruction)
        for attempt in range(5):
            try:
                result = litellm.rerank(
                    model=model_name, query=q,
                    documents=documents, top_n=top_k,
                )
                for rank, r in enumerate(result.results[:top_k]):
                    all_indices[qi, rank] = r["index"]
                    all_scores[qi, rank] = r["relevance_score"]
                break
            except Exception as e:
                if "429" in str(e) or "rate" in str(e).lower():
                    wait = 15 * (attempt + 1)
                    time.sleep(wait)
                else:
                    raise

    return {"indices": all_indices, "scores": all_scores}


async def _arerank_api(queries, documents, top_k, model_name, instruction=""):
    """Async API reranking, one query at a time with rate limit pacing."""
    import litellm
    import time

    n_queries = len(queries)
    all_indices = np.zeros((n_queries, top_k), dtype=np.int64)
    all_scores = np.zeros((n_queries, top_k), dtype=np.float32)

    for qi in range(n_queries):
        q = _apply_instruction(queries[qi], instruction)
        for attempt in range(5):
            try:
                result = await litellm.arerank(
                    model=model_name, query=q,
                    documents=documents, top_n=top_k,
                )
                for rank, r in enumerate(result.results[:top_k]):
                    all_indices[qi, rank] = r["index"]
                    all_scores[qi, rank] = r["relevance_score"]
                break
            except Exception as e:
                if "429" in str(e) or "rate" in str(e).lower():
                    wait = 15 * (attempt + 1)
                    await asyncio.sleep(wait)
                else:
                    raise

    return {"indices": all_indices, "scores": all_scores}


def _rerank_pairs_api(items, model_name, instruction=""):
    """Score pre-paired items using a hosted API, one query at a time."""
    import litellm
    import time

    all_scores = []
    for item in items:
        query, docs = item[0], item[1]
        query = _apply_instruction(query, instruction)
        n_docs = len(docs)
        scores = [0.0] * n_docs
        for attempt in range(5):
            try:
                result = litellm.rerank(
                    model=model_name, query=query,
                    documents=docs, top_n=n_docs,
                )
                for r in result.results:
                    scores[r["index"]] = r["relevance_score"]
                break
            except Exception as e:
                if "429" in str(e) or "rate" in str(e).lower():
                    wait = 15 * (attempt + 1)
                    time.sleep(wait)
                else:
                    raise
        all_scores.append(scores)
    return all_scores


async def _arerank_pairs_api(items, model_name, instruction=""):
    """Async version of _rerank_pairs_api."""
    import litellm

    all_scores = []
    for item in items:
        query, docs = item[0], item[1]
        query = _apply_instruction(query, instruction)
        n_docs = len(docs)
        scores = [0.0] * n_docs
        for attempt in range(5):
            try:
                result = await litellm.arerank(
                    model=model_name, query=query,
                    documents=docs, top_n=n_docs,
                )
                for r in result.results:
                    scores[r["index"]] = r["relevance_score"]
                break
            except Exception as e:
                if "429" in str(e) or "rate" in str(e).lower():
                    wait = 15 * (attempt + 1)
                    await asyncio.sleep(wait)
                else:
                    raise
        all_scores.append(scores)
    return all_scores


def _rerank_pairs_zeroentropy(items, model_name, instruction=""):
    """Score pre-paired items using the ZeroEntropy API."""
    from zeroentropy import ZeroEntropy
    import time

    client = ZeroEntropy()
    all_scores = []
    for item in items:
        query, docs = item[0], item[1]
        query = _apply_instruction_zeroentropy(query, instruction)
        n_docs = len(docs)
        scores = [0.0] * n_docs
        for attempt in range(5):
            try:
                result = client.models.rerank(
                    model=model_name, query=query,
                    documents=docs, top_n=n_docs,
                )
                for r in result.results:
                    scores[r.index] = r.relevance_score
                break
            except Exception as e:
                if "429" in str(e) or "rate" in str(e).lower():
                    wait = 15 * (attempt + 1)
                    time.sleep(wait)
                else:
                    raise
        all_scores.append(scores)
    return all_scores


async def _arerank_pairs_zeroentropy(items, model_name, instruction=""):
    """Async version of _rerank_pairs_zeroentropy."""
    from zeroentropy import AsyncZeroEntropy

    client = AsyncZeroEntropy()
    all_scores = []
    for item in items:
        query, docs = item[0], item[1]
        query = _apply_instruction_zeroentropy(query, instruction)
        n_docs = len(docs)
        scores = [0.0] * n_docs
        for attempt in range(5):
            try:
                result = await client.models.rerank(
                    model=model_name, query=query,
                    documents=docs, top_n=n_docs,
                )
                for r in result.results:
                    scores[r.index] = r.relevance_score
                break
            except Exception as e:
                if "429" in str(e) or "rate" in str(e).lower():
                    wait = 15 * (attempt + 1)
                    await asyncio.sleep(wait)
                else:
                    raise
        all_scores.append(scores)
    return all_scores


def rerank(
    queries: list[str],
    documents: list[str],
    top_k: int = 10,
    *,
    model: str,
    **kwargs,
) -> dict:
    """Rerank documents for each query using a named model key from rerank_models.toml.

    The model key determines the execution mode (api/local/sbatch).
    Any explicit **kwargs override config values.

    Pass slurm_accounting={} to collect Slurm resource accounting data
    (sbatch mode only).

    Returns:
        Dict with "indices" (n_queries, top_k) and "scores" (n_queries, top_k).
    """
    mode, model_name, cfg = _resolve_model_args(rerank_models_config_path, model, kwargs)
    slurm_accounting = cfg.pop("slurm_accounting", None)
    instruction = cfg.pop("instruction", "")

    if mode == "api":
        return _rerank_api(queries, documents, top_k, model_name, instruction=instruction)

    elif mode == "local":
        from llm_runner.rerank import run_rerank
        return run_rerank(queries, documents, top_k, model_name=model_name, **cfg)

    elif mode == "sbatch":
        from isambard_utils.orchestrate import run_remote
        remote_kw = _split_remote_kwargs(cfg)
        cfg["model_name"] = model_name
        cfg["top_k"] = top_k
        result = run_remote(
            "rerank",
            inputs={"queries": queries, "documents": documents},
            config_dict=cfg,
            required_models=[model_name],
            **remote_kw,
        )
        if slurm_accounting is not None and "_slurm_accounting" in result:
            slurm_accounting.update(result.pop("_slurm_accounting"))
        else:
            result.pop("_slurm_accounting", None)
        return result

    else:
        raise ValueError(f"Unknown mode: {mode!r}")

In [ ]:
#|export
def rerank_pairs(
    items,
    *,
    model: str,
    **kwargs,
) -> list[list[float]]:
    """Score grouped (query, documents) pairs using a named model key.

    Unlike rerank() which computes the full cross-product, this scores only
    the specified per-query documents. One GPU call per invocation instead
    of one per query.

    Args:
        items: List of (query, documents) pairs. item[0] is query text,
               item[1] is list of document texts.
        model: Model key from rerank_models.toml.

    Returns:
        list[list[float]]: Scores per item, matching document order.
    """
    mode, model_name, cfg = _resolve_model_args(rerank_models_config_path, model, kwargs)
    slurm_accounting = cfg.pop("slurm_accounting", None)
    instruction = cfg.pop("instruction", "")

    if mode == "api":
        return _rerank_pairs_api(items, model_name, instruction=instruction)

    elif mode == "zeroentropy":
        return _rerank_pairs_zeroentropy(items, model_name, instruction=instruction)

    elif mode == "local":
        from llm_runner.rerank import run_rerank_pairs
        return run_rerank_pairs(items, model_name=model_name, **cfg)

    elif mode == "sbatch":
        from isambard_utils.orchestrate import run_remote
        remote_kw = _split_remote_kwargs(cfg)
        cfg["model_name"] = model_name
        result = run_remote(
            "rerank_pairs",
            inputs={"items": items},
            config_dict=cfg,
            required_models=[model_name],
            **remote_kw,
        )
        if slurm_accounting is not None and "_slurm_accounting" in result:
            slurm_accounting.update(result.pop("_slurm_accounting"))
        else:
            result.pop("_slurm_accounting", None)
        return result["scores"]

    else:
        raise ValueError(f"Unknown mode: {mode!r}")

In [ ]:
#|export
async def arerank_pairs(
    items,
    *,
    model: str,
    **kwargs,
) -> list[list[float]]:
    """Async version of rerank_pairs.

    For local mode, runs in a thread. For sbatch, uses arun_remote.
    """
    mode, model_name, cfg = _resolve_model_args(rerank_models_config_path, model, kwargs)
    slurm_accounting = cfg.pop("slurm_accounting", None)
    instruction = cfg.pop("instruction", "")

    if mode == "api":
        return await _arerank_pairs_api(items, model_name, instruction=instruction)

    elif mode == "zeroentropy":
        return await _arerank_pairs_zeroentropy(items, model_name, instruction=instruction)

    elif mode == "local":
        from llm_runner.rerank import run_rerank_pairs
        return await asyncio.to_thread(run_rerank_pairs, items,
                                       model_name=model_name, **cfg)

    elif mode == "sbatch":
        from isambard_utils.orchestrate import arun_remote
        remote_kw = _split_remote_kwargs(cfg)
        cfg["model_name"] = model_name
        result = await arun_remote(
            "rerank_pairs",
            inputs={"items": items},
            config_dict=cfg,
            required_models=[model_name],
            **remote_kw,
        )
        if slurm_accounting is not None and "_slurm_accounting" in result:
            slurm_accounting.update(result.pop("_slurm_accounting"))
        else:
            result.pop("_slurm_accounting", None)
        return result["scores"]

    else:
        raise ValueError(f"Unknown mode: {mode!r}")

In [ ]:
#|export
async def arerank(
    queries: list[str],
    documents: list[str],
    top_k: int = 10,
    *,
    model: str,
    **kwargs,
) -> dict:
    """Async version of rerank.

    For local mode, runs in a thread. For sbatch, uses arun_remote.

    Pass slurm_accounting={} to collect Slurm resource accounting data
    (sbatch mode only).
    """
    mode, model_name, cfg = _resolve_model_args(rerank_models_config_path, model, kwargs)
    slurm_accounting = cfg.pop("slurm_accounting", None)
    instruction = cfg.pop("instruction", "")

    if mode == "api":
        return await _arerank_api(queries, documents, top_k, model_name, instruction=instruction)

    elif mode == "local":
        from llm_runner.rerank import run_rerank
        return await asyncio.to_thread(run_rerank, queries, documents, top_k,
                                       model_name=model_name, **cfg)

    elif mode == "sbatch":
        from isambard_utils.orchestrate import arun_remote
        remote_kw = _split_remote_kwargs(cfg)
        cfg["model_name"] = model_name
        cfg["top_k"] = top_k
        result = await arun_remote(
            "rerank",
            inputs={"queries": queries, "documents": documents},
            config_dict=cfg,
            required_models=[model_name],
            **remote_kw,
        )
        if slurm_accounting is not None and "_slurm_accounting" in result:
            slurm_accounting.update(result.pop("_slurm_accounting"))
        else:
            result.pop("_slurm_accounting", None)
        return result

    else:
        raise ValueError(f"Unknown mode: {mode!r}")